In [6]:
!pip install -U transformers accelerate bitsandbytes sentencepiece huggingface_hub

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 663.6/663.6 kB 7.5 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 82.4 MB/s  0:00:00 eta 0:00:01
  Attempting uninstall: huggingface_hub
    Found existing installation: huggingface_hub 1.14.0
    Uninstalling huggingface_hub-1.14.0:
      Successfully uninstalled huggingface_hub-1.14.0
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [bitsandbytes] [bitsandbytes]


In [1]:
import torch
import json
import re

In [ ]:
from huggingface_hub import login

login(token="№№№№№№№№№№№№")  # Вставьте ваш токен доступа Hugging Face

In [2]:
from huggingface_hub import whoami
whoami()

{'type': 'user',
 'id': '6991bf496a82ac3be8fffbee',
 'name': 'esworkcolab',
 'fullname': 'ES',
 'isPro': False,
 'avatarUrl': '/avatars/d10df82a3fd28a0e9775400ef71af501.svg',
 'orgs': [],
 'auth': {'type': 'access_token',
  'accessToken': {'displayName': 'stajik',
   'role': 'fineGrained',
   'createdAt': '2026-05-13T17:00:40.836Z',
   'fineGrained': {'canReadGatedRepos': True,
    'global': [],
    'scoped': [{'entity': {'_id': '6991bf496a82ac3be8fffbee',
       'type': 'user',
       'name': 'esworkcolab'},
      'permissions': ['inference.endpoints.infer.write',
       'collection.read',
       'collection.write']}]}}}}

In [3]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

MODEL_ID = "google/medgemma-1.5-4b-it"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4"
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto"
)

model.eval()

Loading weights:   0%|          | 0/883 [00:00<?, ?it/s]

/home/mirea/pas-mri-extractor/.venv/lib/python3.13/site-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


Gemma3ForConditionalGeneration(
  (model): Gemma3Model(
    (vision_tower): SiglipVisionModel(
      (embeddings): SiglipVisionEmbeddings(
        (patch_embedding): Conv2d(3, 1152, kernel_size=(14, 14), stride=(14, 14), padding=valid)
        (position_embedding): Embedding(4096, 1152)
      )
      (encoder): SiglipEncoder(
        (layers): ModuleList(
          (0-26): 27 x SiglipEncoderLayer(
            (layer_norm1): LayerNorm((1152,), eps=1e-06, elementwise_affine=True, bias=True)
            (self_attn): SiglipAttention(
              (k_proj): Linear4bit(in_features=1152, out_features=1152, bias=True)
              (v_proj): Linear4bit(in_features=1152, out_features=1152, bias=True)
              (q_proj): Linear4bit(in_features=1152, out_features=1152, bias=True)
              (out_proj): Linear4bit(in_features=1152, out_features=1152, bias=True)
            )
            (layer_norm2): LayerNorm((1152,), eps=1e-06, elementwise_affine=True, bias=True)
            (mlp): Sigl

In [4]:
def build_prompt(mri_text):
    return f"""
Ты медицинский NLP-экстрактор.

Извлеки признаки из МРТ при подозрении на врастание плаценты.

Ответь ТОЛЬКО одним JSON-объектом.
Не пиши Python.
Не пиши код.
Не пиши markdown.
Не пиши рассуждения.
Не пиши текст до или после JSON.

JSON должен содержать:
- features
- clinical_summary
- clinical_rationale

В features должны быть поля:
invasion_type, invasion_confidence, bladder_involvement, parametrium_involvement,
posterior_wall_involvement, placenta_previa, anterior_placenta,
retroplacental_vessels, lacunae, uterine_wall_thinning,
uterine_hernia_or_bulging, preoperative_bleeding,
previous_cs_count, gestational_week, short_explanation.

Допустимые значения:
- invasion_type: "none", "accreta", "increta", "percreta"
- invasion_confidence: "absent", "possible", "probable", "definite", "unclear"
- остальные признаки: "absent", "possible", "probable", "present"
- previous_cs_count: число или null
- gestational_week: число или null

Правила:
- "нельзя исключить", "возможно", "подозрение" → "possible".
- "вероятно", "соответствует" → "probable".
- если признак явно есть → "present".
- если признака нет в тексте → "absent".
- не выдумывай признаки.

invasion_type:
- percreta: если есть percreta, вовлечение мочевого пузыря, маточно-пузырное пространство не прослеживается/не дифференцируется, выход за пределы матки или параметрий.
- increta: если есть increta, врастание/прорастание в миометрий, миометрий не прослеживается, резкое истончение миометрия с нарушением границы.
- accreta: если есть accreta, плотное прикрепление, поверхностное врастание, врастание без глубины.
- none: если данных за врастание нет или признаков недостаточно.

Отдельные признаки:
- bladder_involvement = present: вовлечение/инвазия мочевого пузыря, деформация стенки мочевого пузыря, маточно-пузырное пространство не прослеживается.
- placenta_previa = present: предлежание, перекрывает внутренний зев, низкое расположение.
- anterior_placenta = present: плацента по передней стенке.
- retroplacental_vessels = present: ретроплацентарные, расширенные, полнокровные сосуды, сосудистый рисунок.
- lacunae = present: лакуны.
- uterine_wall_thinning = present: истончение миометрия/рубца, миометрий не прослеживается.
- preoperative_bleeding = present: кровотечение.

clinical_summary: 1 короткое предложение.
clinical_rationale: 1-2 коротких предложения, только по признакам из текста.

МРТ-текст:
\"\"\"
{mri_text}
\"\"\"

JSON:
""".strip()

In [5]:
def is_bad_json(obj):
    txt = json.dumps(obj, ensure_ascii=False).lower()
    bad_tokens = [
        "...",
        "<одно значение>",
        "none/accreta",
        "absent/possible",
        "python",
        "import json",
        "def "
    ]
    return any(x in txt for x in bad_tokens)


def extract_json_from_response(text):
    text = text.strip()
    text = text.replace("```json", "").replace("```", "").strip()
    text = re.sub(r"<unused\d+>", "", text)
    text = re.sub(r"</unused\d+>", "", text)

    decoder = json.JSONDecoder()

    for i, ch in enumerate(text):
        if ch == "{":
            try:
                obj, end = decoder.raw_decode(text[i:])
                if isinstance(obj, dict) and "features" in obj and not is_bad_json(obj):
                    return obj
            except json.JSONDecodeError:
                continue

    print("RAW MODEL RESPONSE:")
    print(text[:4000])
    raise ValueError("Модель не вернула валидный JSON с features")

In [6]:
def medgemma_extract_mri_json(mri_text, max_new_tokens=700):
    prompt = build_prompt(mri_text)

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=3500
    ).to(model.device)

    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            temperature=0.0,
            pad_token_id=tokenizer.eos_token_id,
            eos_token_id=tokenizer.eos_token_id,
            repetition_penalty=1.05
        )

    generated = output_ids[0][inputs["input_ids"].shape[1]:]
    response = tokenizer.decode(generated, skip_special_tokens=False)

    return extract_json_from_response(response)

In [7]:
def rule_extract_features(mri_text):
    text = str(mri_text).lower().replace("ё", "е")

    features = {
        "invasion_type": "none",
        "invasion_confidence": "absent",
        "bladder_involvement": "absent",
        "parametrium_involvement": "absent",
        "posterior_wall_involvement": "absent",
        "placenta_previa": "present" if re.search(r"предлеж|перекрывает\s+внутренн\w*\s+зев|низк", text) else "absent",
        "anterior_placenta": "present" if re.search(r"передн\w*\s+стенк", text) else "absent",
        "retroplacental_vessels": "present" if re.search(r"ретроплацентарн|расширенн\w*.*сосуд|полнокровн\w*.*сосуд|сосудист", text) else "absent",
        "lacunae": "present" if re.search(r"лакун", text) else "absent",
        "uterine_wall_thinning": "present" if re.search(r"истонч\w*.*миометр|истонч\w*.*рубц|миометр\w*.*не\s+прослеж", text) else "absent",
        "uterine_hernia_or_bulging": "present" if re.search(r"выбух|пролаб|грыж", text) else "absent",
        "preoperative_bleeding": "present" if "кровотеч" in text else "absent",
        "previous_cs_count": None,
        "gestational_week": None,
        "short_explanation": ""
    }

    week = re.search(r"(\d+)\s*недел", text)
    if week:
        features["gestational_week"] = int(week.group(1))

    cs = re.search(r"после\s*(\d+)\s*(?:кс|кесар)", text)
    if cs:
        features["previous_cs_count"] = int(cs.group(1))

    bladder_present = re.search(
        r"вовлеч\w*.*мочев|инвази\w*.*мочев|прорастан\w*.*мочев|"
        r"деформац\w*.*мочев|маточно-пузырн\w*.*не\s+(?:прослеж|дифференц)",
        text
    )

    bladder_possible = re.search(
        r"нельзя\s+исключить.*мочев|возможно.*мочев|подозр\w*.*мочев",
        text
    )

    if bladder_present:
        features["bladder_involvement"] = "present"
    elif bladder_possible:
        features["bladder_involvement"] = "possible"

    if re.search(r"параметр|параметральн", text):
        features["parametrium_involvement"] = "present"

    if re.search(r"задн\w*\s+стенк\w*\s+матк|задн\w*\s+стенк\w*\s+мочев", text):
        features["posterior_wall_involvement"] = "present"

    if re.search(r"\bpercreta\b", text) or bladder_present or re.search(r"за\s+предел\w*.*матк|параметр", text):
        features["invasion_type"] = "percreta"
    elif re.search(r"\bincreta\b|врастан\w*.*миометр|прорастан\w*.*миометр|миометр\w*.*не\s+прослеж", text):
        features["invasion_type"] = "increta"
    elif re.search(r"\baccreta\b|плотн\w*.*прикреп|поверхностн\w*.*врастан|врастан\w*.*плацент", text):
        features["invasion_type"] = "accreta"

    if features["invasion_type"] != "none":
        if re.search(r"нельзя\s+исключить|возможно|подозр", text):
            features["invasion_confidence"] = "possible"
        elif re.search(r"вероят|соответствует", text):
            features["invasion_confidence"] = "probable"
        else:
            features["invasion_confidence"] = "definite"

    found = []

    if features["invasion_type"] != "none":
        found.append(f"тип врастания: {features['invasion_type']}")
    if features["placenta_previa"] == "present":
        found.append("предлежание плаценты")
    if features["anterior_placenta"] == "present":
        found.append("передняя локализация плаценты")
    if features["uterine_wall_thinning"] == "present":
        found.append("истончение миометрия/рубца")
    if features["lacunae"] == "present":
        found.append("плацентарные лакуны")
    if features["retroplacental_vessels"] == "present":
        found.append("расширенные/ретроплацентарные сосуды")
    if features["bladder_involvement"] in ["possible", "present"]:
        found.append(f"мочевой пузырь: {features['bladder_involvement']}")

    features["short_explanation"] = (
        "; ".join(found) if found else "Значимых признаков врастания по тексту не выделено."
    )

    return {"features": features}

In [8]:
def normalize_mri_result(result):
    features = result.get("features", {})

    invasion = features.get("invasion_type", "none")
    confidence = features.get("invasion_confidence", "absent")
    bladder = features.get("bladder_involvement", "absent")
    parametrium = features.get("parametrium_involvement", "absent")
    vessels = features.get("retroplacental_vessels", "absent")
    lacunae = features.get("lacunae", "absent")
    thinning = features.get("uterine_wall_thinning", "absent")
    previa = features.get("placenta_previa", "absent")
    anterior = features.get("anterior_placenta", "absent")
    hernia = features.get("uterine_hernia_or_bulging", "absent")
    bleeding = features.get("preoperative_bleeding", "absent")
    prev_cs = features.get("previous_cs_count", None)

    clinical_score = 0
    reasons = []

    if invasion == "percreta":
        clinical_score += 5
        reasons.append("percreta: +5")
    elif invasion == "increta":
        clinical_score += 3
        reasons.append("increta: +3")
    elif invasion == "accreta":
        clinical_score += 1
        reasons.append("accreta: +1")

    if confidence == "definite" and invasion != "none":
        clinical_score += 2
        reasons.append("явное врастание: +2")
    elif confidence in ["probable", "possible"] and invasion != "none":
        clinical_score += 1
        reasons.append("вероятное/возможное врастание: +1")

    if bladder == "present":
        clinical_score += 3
        reasons.append("вовлечение мочевого пузыря: +3")
    elif bladder in ["possible", "probable"]:
        clinical_score += 2
        reasons.append("возможное вовлечение мочевого пузыря: +2")

    if parametrium == "present":
        clinical_score += 3
        reasons.append("вовлечение параметрия: +3")

    if thinning == "present":
        clinical_score += 1
        reasons.append("истончение миометрия/рубца: +1")

    if hernia == "present":
        clinical_score += 2
        reasons.append("выбухание/грыжа: +2")

    if vessels == "present":
        clinical_score += 1
        reasons.append("расширенные сосуды: +1")

    if lacunae == "present":
        clinical_score += 1
        reasons.append("лакуны: +1")

    if previa == "present":
        clinical_score += 1
        reasons.append("предлежание плаценты: +1")

    if anterior == "present":
        clinical_score += 1
        reasons.append("передняя плацента: +1")

    if bleeding == "present":
        clinical_score += 2
        reasons.append("кровотечение: +2")

    if isinstance(prev_cs, int) and prev_cs >= 2:
        clinical_score += 1
        reasons.append("≥2 КС: +1")

    red_flag = 0

    if bladder in ["present", "probable"]:
        red_flag = 1

    if invasion == "percreta" and bladder in ["possible", "probable", "present"]:
        red_flag = 1

    if invasion == "percreta":
        clinical_score = max(clinical_score, 8)

    if invasion == "percreta" and bladder in ["possible", "probable", "present"]:
        clinical_score = max(clinical_score, 9)

    # более мягкая шкала: increta без мочевого пузыря не улетает сразу в high
    if clinical_score <= 3:
        risk_group = "low"
    elif clinical_score <= 9:
        risk_group = "moderate"
    else:
        risk_group = "high"

    # жёсткие клинические правила
    if invasion == "percreta":
        risk_group = "high"

    if invasion == "increta" and bladder == "absent":
        risk_group = "moderate"

    if invasion == "increta" and risk_group == "low":
        risk_group = "moderate"

    if bladder == "present" and invasion == "percreta":
        risk_group = "high"

    if risk_group == "low":
        blood_risk, vascular_risk, bladder_risk = 10, 10, 5
        blood_loss_range = "500–1000 мл"
    elif risk_group == "moderate":
        blood_risk, vascular_risk, bladder_risk = 35, 25, 15
        blood_loss_range = "1000–1500 мл"
    else:
        blood_risk, vascular_risk, bladder_risk = 70, 50, 30
        blood_loss_range = "1500–2500 мл"

    if invasion == "percreta":
        blood_risk = max(blood_risk, 70)
        vascular_risk = max(vascular_risk, 50)
        blood_loss_range = "1500–3000 мл"

    if invasion == "increta" and bladder == "absent":
        blood_risk = min(max(blood_risk, 40), 50)
        vascular_risk = min(max(vascular_risk, 25), 35)
        bladder_risk = min(bladder_risk, 15)
        blood_loss_range = "1000–1500 мл"

    if bladder == "present":
        bladder_risk = max(bladder_risk, 40)
        blood_loss_range = "1500–3000 мл"
    elif bladder in ["possible", "probable"]:
        bladder_risk = max(bladder_risk, 30)

    if vessels == "present":
        blood_risk += 5
        vascular_risk += 5

    if lacunae == "present":
        blood_risk += 5

    if previa == "present":
        blood_risk += 5

    if anterior == "present":
        vascular_risk += 5

    if thinning == "present":
        blood_risk += 5

    # ограничение, чтобы moderate increta не превращалась в 85%
    if invasion == "increta" and bladder == "absent":
        blood_risk = min(blood_risk, 50)
        vascular_risk = min(vascular_risk, 40)
        bladder_risk = min(bladder_risk, 15)
        blood_loss_range = "1000–1500 мл"

    blood_risk = int(min(blood_risk, 85))
    vascular_risk = int(min(vascular_risk, 75))
    bladder_risk = int(min(bladder_risk, 65))

    if risk_group == "low":
        readiness_level = "1"
        readiness_text = "Уровень 1: низкий риск, стандартная бригада, обычная подготовка."
    elif risk_group == "moderate":
        readiness_level = "2"
        readiness_text = "Уровень 2: умеренный риск, усиленная подготовка, запас компонентов крови, сосудистый хирург по вызову."
    else:
        readiness_level = "3"
        readiness_text = "Уровень 3: высокий риск, мультидисциплинарная команда, сосудистый хирург/уролог заранее, готовность к расширенной операции."

    result["score"] = {
        "clinical_score": clinical_score,
        "risk_group": risk_group,
        "red_flag": red_flag,
        "score_reasons": "; ".join(reasons) if reasons else "значимых признаков высокого риска не выявлено"
    }

    result["predicted_risks"] = {
        "massive_blood_loss_over_1500_ml_percent": blood_risk,
        "estimated_blood_loss_ml_range": blood_loss_range,
        "vascular_intervention_percent": vascular_risk,
        "bladder_involvement_percent": bladder_risk,
        "risk_summary_text": (
            f"Риск массивной кровопотери >1500 мл: {blood_risk}%; "
            f"прогнозируемый объём кровопотери: {blood_loss_range}; "
            f"риск необходимости сосудистого вмешательства: {vascular_risk}%; "
            f"риск вовлечения мочевого пузыря: {bladder_risk}%"
        )
    }

    result["recommendation"] = {
        "readiness_level": readiness_level,
        "readiness_text": readiness_text
    }

    result["computed_rationale"] = (
        f"Уровень готовности выбран на основании признаков: {result['score']['score_reasons']}. "
        f"Оценочные риски: кровопотеря >1500 мл — {blood_risk}%, "
        f"прогнозируемый объём кровопотери — {blood_loss_range}, "
        f"сосудистое вмешательство — {vascular_risk}%, "
        f"вовлечение мочевого пузыря — {bladder_risk}%."
    )

    return result

In [9]:
def print_mri_risk_result(result):
    print("=" * 60)
    print("JSON признаки:")
    print(json.dumps(result["features"], ensure_ascii=False, indent=2))

    if result.get("clinical_summary"):
        print("\nКлинический вывод:")
        print(result["clinical_summary"])

    if result.get("clinical_rationale"):
        print("\nОбоснование:")
        print(result["clinical_rationale"])

    if result.get("computed_rationale"):
        print("\nРасчётное обоснование:")
        print(result["computed_rationale"])

    print("\nБалльная оценка:")
    print(json.dumps(result["score"], ensure_ascii=False, indent=2))

    risks = result["predicted_risks"]
    rec = result["recommendation"]

    print("\nРиски:")
    print(f"Риск массивной кровопотери >1500 мл: {risks['massive_blood_loss_over_1500_ml_percent']}%")
    print(f"Прогнозируемый объём кровопотери: {risks['estimated_blood_loss_ml_range']}")
    print(f"Риск необходимости сосудистого вмешательства: {risks['vascular_intervention_percent']}%")
    print(f"Риск вовлечения мочевого пузыря: {risks['bladder_involvement_percent']}%")

    print("\nИтог:")
    print(f"Score: {result['score']['clinical_score']}")
    print(f"Risk group: {result['score']['risk_group']}")
    print(f"Red flag: {result['score']['red_flag']}")
    print(f"Уровень готовности: {rec['readiness_level']}")
    print(f"Рекомендация: {rec['readiness_text']}")


def predict_full_mri_medgemma(mri_description, mri_conclusion=""):
    full_text = f"{mri_description}\n{mri_conclusion}"

    try:
        result = medgemma_extract_mri_json(full_text)

        if not isinstance(result, dict) or "features" not in result or is_bad_json(result):
            print("MedGemma вернула некорректный JSON, использую fallback rules.")
            result = rule_extract_features(full_text)

    except Exception as e:
        print("Ошибка MedGemma:", e)
        print("Использую fallback rules.")
        result = rule_extract_features(full_text)

    result = normalize_mri_result(result)
    print_mri_risk_result(result)

    return result

In [10]:
mri_description = """
Плацента расположена по задней стенке матки.
Контуры ровные, структура однородная.
Граница между плацентой и миометрием четкая.
Миометрий сохранен, истончения не выявлено.
Ретроплацентарные сосуды не расширены.
Маточно-пузырное пространство дифференцируется.
"""

mri_conclusion = """
МР-признаков врастания плаценты не выявлено.
"""

result = predict_full_mri_medgemma(mri_description, mri_conclusion)

[transformers] The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
/home/mirea/pas-mri-extractor/.venv/lib/python3.13/site-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


JSON признаки:
{
  "invasion_type": "none",
  "invasion_confidence": "absent",
  "bladder_involvement": "absent",
  "parametrium_involvement": "absent",
  "posterior_wall_involvement": "absent",
  "placenta_previa": "absent",
  "anterior_placenta": "absent",
  "retroplacental_vessels": "absent",
  "lacunae": "absent",
  "uterine_wall_thinning": "absent",
  "uterine_hernia_or_bulging": "absent",
  "preoperative_bleeding": "absent",
  "previous_cs_count": null,
  "gestational_week": null,
  "short_explanation": "Плацента расположена по задней стенке матки. Контуры ровные, структура однородная. Граница между плацентой и миометрием четкая. Миометрий сохранен, истончения не выявлено. Ретроплацентарные сосуды не расширены. Маточно-пузырное пространство дифференцируется."
}

Клинический вывод:
МРТ-признаков врастания плаценты не выявлено.

Обоснование:
МР-признаков врастания плаценты не выявлено.

Расчётное обоснование:
Уровень готовности выбран на основании признаков: значимых признаков высо

In [11]:
mri_description = """
Плацента расположена по передней стенке матки, частично перекрывает внутренний зев.
В области рубца отмечается истончение миометрия до 2–3 мм.
Граница между плацентой и миометрием местами снижена.
Определяются единичные плацентарные лакуны.
Ретроплацентарные сосуды умеренно расширены.
Маточно-пузырное пространство дифференцируется.
"""

mri_conclusion = """
МР-признаки placenta increta.
"""

result = predict_full_mri_medgemma(mri_description, mri_conclusion)

JSON признаки:
{
  "invasion_type": "increta",
  "invasion_confidence": "probable",
  "bladder_involvement": "absent",
  "parametrium_involvement": "absent",
  "posterior_wall_involvement": "absent",
  "placenta_previa": "present",
  "anterior_placenta": "present",
  "retroplacental_vessels": "present",
  "lacunae": "present",
  "uterine_wall_thinning": "present",
  "uterine_hernia_or_bulging": "absent",
  "preoperative_bleeding": "absent",
  "previous_cs_count": null,
  "gestational_week": null,
  "short_explanation": "Плацента расположена по передней стенке матки, частично перекрывает внутренний зев. В области рубца отмечается истончение миометрия до 2–3 мм. Граница между плацентой и миометрием местами снижена. Определяются единичные плацентарные лакуны. Ретроплацентарные сосуды умеренно расширены. Маточно-пузырное пространство дифференцируется."
}

Клинический вывод:
МРТ-признаки placenta increta.

Обоснование:
Плацента расположена по передней стенке матки, частично перекрывает внут

In [12]:
mri_description = """
МРТ органов малого таза выполнено на 35 неделе беременности.

Плацента расположена по передней стенке матки, полностью перекрывает внутренний зев.
В области послеоперационного рубца определяется выраженное истончение миометрия,
местами миометрий не прослеживается.

Отмечается нарушение дифференциации между плацентой и стенкой матки.
Определяются множественные плацентарные лакуны и выраженные ретроплацентарные сосуды.

Маточно-пузырное пространство не прослеживается.
Задняя стенка мочевого пузыря деформирована и тесно прилежит к зоне плацентарной площадки.
"""

mri_conclusion = """
МР-признаки placenta percreta с вовлечением мочевого пузыря.
"""

result = predict_full_mri_medgemma(mri_description, mri_conclusion)

JSON признаки:
{
  "invasion_type": "percreta",
  "invasion_confidence": "definite",
  "bladder_involvement": "present",
  "parametrium_involvement": "absent",
  "posterior_wall_involvement": "absent",
  "placenta_previa": "present",
  "anterior_placenta": "present",
  "retroplacental_vessels": "present",
  "lacunae": "present",
  "uterine_wall_thinning": "present",
  "uterine_hernia_or_bulging": "absent",
  "preoperative_bleeding": "absent",
  "previous_cs_count": null,
  "gestational_week": 35,
  "short_explanation": "МРТ органов малого таза выполнено на 35 неделе беременности. Плацента расположена по передней стенке матки, полностью перекрывает внутренний зев. В области послеоперационного рубца определяется выраженное истончение миометрия, местами миометрий не прослеживается. Отмечается нарушение дифференциации между плацентой и стенкой матки. Определяются множественные плацентарные лакуны и выраженные ретроплацентарные сосуды. Маточно-пузырное пространство не прослеживается. Задняя